In [55]:
import pandas as pd
import chainladder as cl
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

auto_bi = cl.load_sample('friedland_auto_bi_insurer')

# Exhibit — Reported / Paid / reform factors (1)–(2) / adjusted claims / premium / LDF (7)–(8) / ultimate / claim ratio
exhibit = pd.DataFrame(
    [
        [2000, 10_000_000, 9_500_000, 1.005, 1.050, 10_050_000, 9_975_000, 10_012_500, 24_000_000, 2.954, 0.670, 19_816_540, 0.830],
        [2001, 8_000_000, 7_200_000, 1.020, 1.150, 8_160_000, 8_280_000, 8_220_000, 18_000_000, 2.580, 0.670, 14_209_092, 0.790],
        [2002, 9_400_000, 7_600_000, 1.030, 1.250, 9_682_000, 9_500_000, 9_591_000, 19_000_000, 2.253, 0.670, 14_477_710, 0.760],
        [2003, 15_600_000, 7_800_000, 1.100, 1.350, 17_160_000, 10_530_000, 13_845_000, 23_000_000, 1.968, 0.670, 18_255_463, 0.790],
        [2004, 16_500_000, 11_200_000, 1.200, 1.750, 19_800_000, 19_600_000, 19_700_000, 32_000_000, 1.719, 0.750, 25_398_225, 0.790],
        [2005, 18_500_000, 10_200_000, 1.400, 2.500, 25_900_000, 25_500_000, 25_700_000, 47_000_000, 1.501, 1.000, 38_575_700, 0.820],
        [2006, 16_500_000, 6_000_000, 1.800, 5.000, 29_700_000, 30_000_000, 29_850_000, 50_000_000, 1.311, 1.000, 39_133_350, 0.780],
        [2007, 14_000_000, 3_000_000, 2.900, 15.000, 40_600_000, 45_000_000, 42_800_000, 57_000_000, 1.145, 1.000, 49_006_000, 0.860],
        [2008, 8_700_000, 750_000, 4.000, 90.000, None, None, None, None, None, None, None, None],
    ],
    columns=[
        'Accident Year',
        'Reported',
        'Paid',
        '(1)',
        '(2)',
        'Reported to 7/1/08',
        'Paid Reform',
        'Reported Paid Claims',
        'Premium',
        '(7)',
        '(8)',
        'Claims',
        'Claim Ratio',
    ],
)
exhibit

,Accident Year,Reported,Paid,(1),(2),Reported to 7/1/08,Paid Reform,Reported Paid Claims,Premium,(7),(8),Claims,Claim Ratio
0,2000,10000000,9500000,1.005,1.05,10050000.0,9975000.0,10012500.0,24000000.0,2.954,0.67,19816540.0,0.83
1,2001,8000000,7200000,1.020,1.15,8160000.0,8280000.0,8220000.0,18000000.0,2.580,0.67,14209092.0,0.79
2,2002,9400000,7600000,1.030,1.25,9682000.0,9500000.0,9591000.0,19000000.0,2.253,0.67,14477710.0,0.76
3,2003,15600000,7800000,1.100,1.35,17160000.0,10530000.0,13845000.0,23000000.0,1.968,0.67,18255463.0,0.79
4,2004,16500000,11200000,1.200,1.75,19800000.0,19600000.0,19700000.0,32000000.0,1.719,0.75,25398225.0,0.79
5,2005,18500000,10200000,1.400,2.50,25900000.0,25500000.0,25700000.0,47000000.0,1.501,1.00,38575700.0,0.82
6,2006,16500000,6000000,1.800,5.00,29700000.0,30000000.0,29850000.0,50000000.0,1.311,1.00,39133350.0,0.78
7,2007,14000000,3000000,2.900,15.00,40600000.0,45000000.0,42800000.0,57000000.0,1.145,1.00,49006000.0,0.86
8,2008,8700000,750000,4.000,90.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [56]:
# Page 140, Exhibit 1 column 2
auto_bi = cl.load_sample("friedland_auto_bi_insurer")
auto_bi["Reported Claims"].latest_diagonal

,2008
2000,"10,000,000"
2001,"8,000,000"
2002,"9,400,000"
2003,"15,600,000"
2004,"16,500,000"
2005,"18,500,000"
2006,"16,500,000"
2007,"14,000,000"
2008,"8,700,000"


In [57]:
# Page 140, Exhibit 1 column 3
auto_bi["Paid Claims"].latest_diagonal

,2008
2000,"9,500,000"
2001,"7,200,000"
2002,"7,600,000"
2003,"7,800,000"
2004,"11,200,000"
2005,"10,200,000"
2006,"6,000,000"
2007,"3,000,000"
2008,"750,000"


In [58]:
# Page 140, Exhibit 1 column 4-5
# Direct age(in months): value representation per LOB
reported_pattern = {12: 4.0, 24: 2.9, 36: 1.8, 48: 1.4, 60: 1.2, 72: 1.1, 84: 1.03, 96: 1.02, 108: 1.005}
paid_pattern = {12: 90.0, 24: 15.0, 36: 5.0, 48: 2.5, 60: 1.75, 72: 1.35, 84: 1.25, 96: 1.15, 108: 1.05}

In [59]:
# Page 140, Exhibit 1 column 6
Reported_BI = cl.DevelopmentConstant(patterns=reported_pattern, style='cdf').fit_transform(auto_bi["Reported Claims"])
reported_ultimate = cl.Chainladder().fit(Reported_BI).ultimate_
reported_ultimate

,2261
2000,"10,050,000"
2001,"8,160,000"
2002,"9,682,000"
2003,"17,160,000"
2004,"19,800,000"
2005,"25,900,000"
2006,"29,700,000"
2007,"40,600,000"
2008,"34,800,000"


In [60]:
# Page 140, Exhibit 1 column 7
Paid_BI = cl.DevelopmentConstant(patterns=paid_pattern, style='cdf').fit_transform(auto_bi["Paid Claims"])
paid_ultimate = cl.Chainladder().fit(Paid_BI).ultimate_
paid_ultimate

,2261
2000,"9,975,000"
2001,"8,280,000"
2002,"9,500,000"
2003,"10,530,000"
2004,"19,600,000"
2005,"25,500,000"
2006,"30,000,000"
2007,"45,000,000"
2008,"67,500,000"


In [61]:
# Page 140, Exhibit 1 column 8
inital_selected_ultiamte_claims = (reported_ultimate + paid_ultimate)/2
inital_selected_ultiamte_claims

,2261
2000,"10,012,500"
2001,"8,220,000"
2002,"9,591,000"
2003,"13,845,000"
2004,"19,700,000"
2005,"25,700,000"
2006,"29,850,000"
2007,"42,800,000"
2008,"51,150,000"


In [62]:
# Page 140, Exhibit 1 column 9
auto_bi["Earned Premium"].latest_diagonal

,2008
2000,"24,000,000"
2001,"18,000,000"
2002,"19,000,000"
2003,"23,000,000"
2004,"32,000,000"
2005,"47,000,000"
2006,"50,000,000"
2007,"57,000,000"
2008,"62,000,000"


In [63]:
# Page 140, Exhibit 1 column 10
trend_factors = np.round(cl.Trend(trends=[0.145], dates=[("2008-12-31", "2000-01-01")]).fit(auto_bi["Earned Premium"]).trend_.latest_diagonal,3)
trend_factors

,2008
2000,2.9540
2001,2.5800
2002,2.2530
2003,1.9680
2004,1.7190
2005,1.5010
2006,1.3110
2007,1.1450
2008,1.0000


In [64]:
# Page 140, Exhibit 1 column 11
tort_factors = [0.670, 0.670, 0.670, 0.670, 0.750, 1.0, 1.0, 1.0, 1.0]
tort_factors

[0.67, 0.67, 0.67, 0.67, 0.75, 1.0, 1.0, 1.0, 1.0]

In [65]:
# Page 140, Exhibit 1 column 12
trended_adj_ultimate_claims = trend_factors * inital_selected_ultiamte_claims * np.array(tort_factors).reshape(1, 1, -1, 1)
trended_adj_ultimate_claims

,2008
2000,"19,816,540"
2001,"14,209,092"
2002,"14,477,710"
2003,"18,255,463"
2004,"25,398,225"
2005,"38,575,700"
2006,"39,133,350"
2007,"49,006,000"
2008,"51,150,000"


In [68]:
# Page 140, Exhibit 1 column 13
trended_adjusted_claim_ratio = np.round(trended_adj_ultimate_claims/auto_bi["Earned Premium"].latest_diagonal,2)
trended_adjusted_claim_ratio

,2008
2000,0.8300
2001,0.7900
2002,0.7600
2003,0.7900
2004,0.7900
2005,0.8200
2006,0.7800
2007,0.8600
2008,0.8200


In [ ]:
# Page 140, Exhibit 1 item 14
# Average trended adjusted claim ratio for accident years 2000-2005 (origin indices 0:6)
print("Average 2000 to 2005:", np.round(trended_adjusted_claim_ratio.iloc[:, :, 0:6, :].mean(),3))
loss_ratios_00_05 = trended_adjusted_claim_ratio.iloc[:, :, 0:6, :].values.flatten()
print("Average 2000 to 2005 Ex High Ex Low:", np.round((loss_ratios_00_05.sum() - loss_ratios_00_05.max() - loss_ratios_00_05.min()) / (len(loss_ratios_00_05) - 2), 3))
print("Average 2001 to 2006:", np.round(trended_adjusted_claim_ratio.iloc[:, :, 1:7, :].mean(),3))
loss_ratios_01_06 = trended_adjusted_claim_ratio.iloc[:, :, 1:7, :].values.flatten()
print(loss_ratios_01_06)
print("Average 2000 to 2005 Ex High Ex Low:", np.round((loss_ratios_01_06.sum() - loss_ratios_01_06.max() - loss_ratios_01_06.min()) / (len(loss_ratios_01_06) - 2), 3)) # small rounding error due to python's 64 bit float storage

Average 2000 to 2005: 0.797
Average 2000 to 2005 Ex High Ex Low: 0.798
Average 2001 to 2006: 0.788
[0.79 0.76 0.79 0.79 0.82 0.78]
Average 2000 to 2005 Ex High Ex Low: 0.787


In [71]:
# Page 140, Exhibit 1 item 15
selected_claim_ratio = 0.80
selected_claim_ratio

0.8